# RAG(Retrieval Augmented Generation)
- [RAG](https://python.langchain.com/v0.1/docs/modules/data_connection/)은 *Retrieval Augmented Generation*의 약자로, **검색 기반 생성 기법**을 의미한다. 이 기법은 LLM이 특정 문서에 기반하여 보다 정확하고 신뢰할 수 있는 답변을 생성할 수 있도록 돕는다.     
- 사용자의 질문에 대해 자체적으로 구축한 데이터베이스(DB)나 외부 데이터베이스에서 질문과 관련된 문서를 검색하고, 이를 질문과 함께 LLM에 전달한다.
- LLM은 같이 전달된 문서를 바탕으로 질문에 대한 답변을 생성한다. 
- 이를 통해 LLM이 학습하지 않은 내용도 다룰 수 있으며, 잘못된 정보를 생성하는 환각 현상(*hallucination*)을 줄일 수 있다.

## RAG와 파인튜닝(Fine Tuning) 비교

### 파인튜닝(Fine Tuning)

- **정의**: 사전 학습(pre-trained)된 LLM에 특정 도메인의 데이터를 추가로 학습시켜 해당 도메인에 특화된 맞춤형 모델로 만드는 방식이다.
- **장점**
  - 특정 도메인에 최적화되어 높은 정확도와 성능을 낼 수 있다.
- **단점**
  - 모델 재학습에 많은 시간과 자원이 필요하다.
  - 새로운 정보가 반영되지 않으며, 이를 위해서는 다시 학습해야 한다.

### RAG

- **정의**: 모델을 다시 학습시키지 않고, 외부 지식 기반에서 정보를 검색하여 실시간으로 답변에 활용하는 방식이다.
- **장점**
  - 최신 정보를 쉽게 반영할 수 있다.
  - 모델을 수정하지 않아도 되므로 효율적이다.
- **단점**
  - 검색된 문서의 품질에 따라 답변의 정확성이 달라질 수 있다.
  - 검색 시스템 구축이 필요하다.

## 정리

| 항목       | 파인튜닝 | RAG |
| -------- | ---- | --- |
| 도메인 최적화  | 가능   | 제한적 |
| 최신 정보 반영 | 불가능  | 가능  |
| 구현 난이도   | 높음   | 보통  |
| 유연성      | 낮음   | 높음  |

- LLM은 학습 당시의 데이터만을 기반으로 작동하므로 최신 정보나 기업 내부 자료와 같은 특정한 지식 기반에 접근할 수 없다.
- 파인튜닝은 시간과 비용이 많이 들고 유지보수가 어렵다.
-	반면, RAG는 기존 LLM을 변경하지 않고도 외부 문서를 통해 그 한계를 보완할 수 있다.
- RAG는 특히 빠르게 변화하는 정보를 다루는 분야(예: 기술 지원, 뉴스, 법률 등)에서 유용하게 활용된다. 반면, 정적인 정보에 대해 높은 정확도가 필요한 경우에는 파인튜닝이 효과적이다.


## RAG 작동 단계
- 크게 "**정보 저장(인덱싱)**", "**검색**, **생성**"의 단계로 나눌 수 있다.
  
### 1. 정보 저장(인덱싱)
RAG는 사전에 정보를 가공하여 **벡터 데이터베이스**(Vector 저장소)에 저장해 두고, 나중에 검색할 수 있도록 준비한다. 이 단계는 다음과 같은 과정으로 이루어진다.

1. **Load (불러오기)**
   - 답변시 참조할 사전 정보를 가진 데이터들을 불러온다.
2. **Split/Chunking (문서 분할)**
   - 긴 텍스트를 일정한 길이의 작은 덩어리(*chunk*)로 나눈다.
   - 이렇게 해야 검색과 생성의 정확도를 높일 수 있다.
3. **Embedding (임베딩)**
   - 각 텍스트 조각을 **임베딩 벡터**로 변환한다.
   - 임베딩 벡터는 그 문서의 의미를 벡터화 한 것으로 질문과 유사한 문서를 찾을 때 인덱스로 사용된다.
4. **Store (저장)**
   - 임베딩된 벡터를 **벡터 데이터베이스**(벡터 저장소)에 저장한다.
   - 벡터 데이터베이스는 유사한 질문이나 문장을 빠르게 찾을 수 있도록 특화된 데이터 저장소이다.
   
![rag](figures/rag1.png)

### 2. 검색, 생성

사용자가 질문을 하면 다음과 같은 절차로 답변이 생성된다.
1. **Retrieve (검색)**
   - 사용자의 질문을 임베딩한 후, 이 질문 벡터와 유사한 context 벡터를 벡터 데이터베이스에서 검색하여 찾는다.
2. **Query (질의 생성)**
   - 벡터 데이터베이스에서 검색된 문서 조각과 사용자의 질문을 함께 **프롬프트**(prompt)로 구성하여 LLM에 전달한다.
3. **Generation (응답 생성)**
   - LLM은 받은 프롬프트에 대한 응답을 생성한다.
   
- **RAG 흐름**
  
![Retrieve and Generation](figures/rag2.png)


# Document Loader
- LLM에게 질의할 때 같이 제공할 Data들을 저장하기 위해 먼저 읽어들인다.(Load)
- 데이터 Resouce는 다양하다.
    - 데이터를 로드(load)하는 방식은 저장된 위치와 형식에 따라 다양하다. 
      - 로컬 컴퓨터(Local Computer)에 저장된 문서
        - 예: CSV, Excel, JSON, TXT 파일 등
      - 데이터베이스(Database)에 저장된 데이터셋
      - 인터넷에 존재하는 데이터
        - 예: 웹에 공개된 API, 웹 페이지에 있는 데이터, 클라우드 스토리지에 저장된 파일 등

![rag_load](figures/rag_load.png)

- 다양한 문서 형식(format)에 맞춰 읽어오는 다양한 **document loader** 들을 Langchain에서 지원한다.
    - 다양한 Resource들로 부터 데이터를 읽기 위해서는 다양한 라이브러리를 이용해 서로 다른 방법으로 읽어야 한다.
    - Langchain은 데이터를 읽는 다양한 방식의 코드를 하나의 interface로 사용 할 수 있도록 지원한다.
        - https://python.langchain.com/docs/how_to/#document-loaders
    - 다양한 3rd party library(ppt, github 등등 다양한 3rd party lib도 있음. )들과 연동해 다양한 Resource로 부터 데이터를 Loading 할 수 있다.
        - https://python.langchain.com/docs/integrations/document_loaders/
- **모든 document loader는 기본적으로 동일한 interface(사용법)로 호출할 수있다.**
- **반환타입**
    - **list[Document]**
    - Load 한 문서는 Document객체에 정보들을 넣는다. 여러 문서를 읽을 수 있기 대문에 list에 묶어서 반환한다.
        - **Document 속성**
            - page_content: 문서의 내용
            - metadata(option): 문서에 대한 메타데이터(정보)를 dict 형태로 저장한다. 
            - id(option): 문서의 고유 id
     
- **주의**
    - Langchain을 이용해 RAG를 구현할 때 **꼭 Langchain의 DocumentLoader를 사용해야 하는 것은 아니다.**
    - DocumentLoader는 데이터를 읽어오는 것을 도와주는 라이브러리일 뿐이다. 다른 라이브러리를 이용해서 읽어 들여도 상관없다. 

## 주요 Document Loader

### Text file
- TextLoader 이용

In [ ]:
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

loader = TextLoader(path, encoding='utf-8')

docs = loader.load()

print(type(docs), len(docs))

In [ ]:
docs
# docs[0]

In [ ]:
# 문서 내용
print(docs[0].page_content[:100])

In [ ]:
print(docs[0].metadata)
docs[0].metadata['category'] = "스포츠"
docs[0].metadata['tag']=['올림픽', '동계올림픽', "IOC"]
print(docs[0].metadata)

### PDF
- PyPDF, Pymupdf 등 다양한 PDF 문서를 읽어들이는 파이썬의  3rd party library들을 이용해 pdf 문서를 Load 한다.
    - https://python.langchain.com/docs/integrations/document_loaders/#pdfs
- 각 PDF Loader 특징
    -  PyMuPDFLoader
        -   텍스트 뿐 아니라 이미지, 주석등의 정보를 추출하는데 성능이 좋다.
        -   PyMuPDF 라이브러리 기반
    - PyPDFLoader
        - 텍스트를 빠르게 추출 할 수있다.
        - PyPDF2 라이브러리 기반. 경량 라이브러리로 빠르고 큰 파일도 효율적으로 처리한다.
    - PDFPlumberLoader
        - 표와 같은 복잡한 구조의 데이터 처리하는데 강력한 성능을 보여준다. 텍스트, 이미지, 표 등을 모두 추출할 수 있다. 
        - PDFPlumber 라이브러리 기반
- 설치 패키지
    - DocumentLoader와 연동하는 라이브러리들을 설치 해야 한다.
    - `pip install pypdf -qU`
    - `pip install pymupdf -qU`
    - `pip install pdfplumber -qU`

In [ ]:
# !uv pip install pypdf pymupdf pdfplumber 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader

path = "data/novel/동백꽃_김유정.pdf"
loader = PyPDFLoader(path, mode="single") 
# loader = PyMuPDFLoader(path)
# loader = PDFPlumberLoader(path)

docs = loader.load()

In [ ]:
docs[0].metadata

In [ ]:
len(docs)

In [ ]:
print(len(docs))
for doc in docs:
    doc.metadata['author'] = '김유정'

In [ ]:
docs[0].metadata

In [ ]:
print(docs[2].page_content[:500])

### CSVLoader

In [ ]:
from langchain_community.document_loaders import CSVLoader

path = "data/boston_hosing.csv"
loader = CSVLoader(path)
docs = loader.load()
print(len(docs))

In [ ]:
docs[10].metadata

In [ ]:
print(docs[10].page_content)

### Web 문서 로드

#### WebBaseLoader를 이용해 Web 문서로딩

requests와 BeautifulSoup을 이용해 web 페이지의 내용을 크롤링해서 Document로 loading한다.

- 주요 파라미터
  - **web_paths***: str | list[str]
    - 크롤링할 대상 URL
  - **requests_kwargs**: dict
    - requests.get() 에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    - headers, cookies, verify 등 설정 전달
  - **header_template**: dict
    - HTTP Header 에 넣을 값을 dict 로 전달.
  - **encoding**
    - requests의 응답 encoding을 설정 (bs_kwargs의 from_encoding 보다 상위에서 적용됨)
  - **bs_kwargs**
    - BeautifulSoup initializer에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    -  주요 옵션
       - **parse_only**: 요청 페이지에서 특정 요소만 선택해서 가져오기. **SoupStrainer를 사용**한다.
         - BeautifulSoup의 `SoupStrainer` 를 이용해 페이지의 일부분만 가져오기
           - 웹 페이지를 파싱(parse, 구조 분석)할 때, 페이지 전체가 아닌 특정 부분만 필요한 경우가 많다. BeautifulSoup 라이브러리의 SoupStrainer를 사용하면, 원하는 태그나 속성이 있는 요소만 골라서 파싱할 수 있다.
           - BeautifulSoup("html문서", parse_only=Strainer객체)
               - Strainer객체에 지정된 영역에서만 내용 찾는다.
           - `SoupStrainer("태그명")`, `SoupStrainer(["태그명", "태그명"])`
             - 지정한 태그 만 조회
           - `SoupStrainer(name="태그명", attrs={속성명:속성값})`
             -  지정한 태그 중 속성명=속성값인 것만 조회
        - **from_encoding**: Encoding 설정 
          - "from_encoding":"utf-8"
   - **bs_get_text_kwargs**:
     - BeautifulSoup객체.get_text() 에 전달할 파라미터 dict로 전달. (key: parameter변수명, value: 전달할 값)
     - **RAG 구축시 `separator` 와 `strip=True` 으로 설정하는 것이 좋다.** (RAG 품질을 위해 강력히 권장되는 설정이다.)
       -  get_text() 는 기본적으로 태그를 제거하고 텍스트만 이어 붙여 반환한다. `separator=구분자문자` 를 지정하여 추출된 텍스트 요소들 사이에 원하는 구분자를 지정할 수있다. `\n` 을 구분자로 사용하면 텍스트 블록 사이에 줄바꿈이 들어가 **문단의 구조를 어느정도 살릴 수 있다.**
       -  웹 문서의 줄바꿈도 포함해서 읽기 때문에 공백과 줄바꿈이 혼재된 상태로 반환된다. `strip=True`로 설정하면 추출된 문자 앞뒤의 공백 문자들을 제거할 수있다.

In [ ]:
# !uv pip install requests beautifulsoup4 lxml

In [ ]:
from bs4 import BeautifulSoup
html_txt = """
<html>
<body>
<p><b>제목</b>  <span>내용</span></p>
<p>다음문단</p>
<div>다음 내용</div>
</body>
</html>"""
soup = BeautifulSoup(html_txt)
# 태그 빼고 text만 추출 -> get_text()
txt1 = soup.get_text()
print("-------기본-----")
print(txt1)

txt2 = soup.get_text(strip=True) # 좌우 공백문자(공백, 엔터) 제거
print("------strip=True-------")
print(txt2)

txt3 = soup.get_text(strip=True, separator="\n\n") # 각 태그의 text를 지정한 구분자로 나눈다.
print("----------strip=True, separator=----------")
print(txt3)

In [ ]:
import os 
# 크롬: my user agent 로 검색
# USER AGENT를 환경변수에 등록
os.environ['USER_AGENT'] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

urls= [
    "https://m.sports.naver.com/kbaseball/article/076/0004357999",
    "https://m.sports.naver.com/kfootball/article/425/0000177935"
]


loader = WebBaseLoader(
    web_path=urls,
    default_parser="lxml"
)

docs = loader.load()
print(len(docs))

In [ ]:
from pprint import pprint
pprint(docs[0].metadata)

In [ ]:
print(docs[0].page_content)

In [ ]:
from bs4 import SoupStrainer
# SoupStrainer 이용

loader2 = WebBaseLoader(
    web_path=urls,
    bs_kwargs={
        "parse_only":SoupStrainer(attrs={"class":['_article_content']})
    },
    bs_get_text_kwargs={
        "separator":"\n", "strip":True
    }

)
docs2 = loader2.load()
len(docs2)

In [ ]:
print(docs2[0].metadata)

In [ ]:
print(docs2[0].page_content)

#### RecursiveUrlLoader

- 주어진 URL에서 시작하여 그 페이지 안의 내부 링크를 재귀적으로 따라가며 여러 웹 문서를 자동 수집하여 로드한다.
  - 시작 url을 요청/페이지를 파싱 한 뒤에 `<a href>` 들을 수집하고 그 페이지들을 요청/페이지 파싱을 한다. 
- WebBaseLoader가 단일 페이지(단일 URL) 단위라면 RecursiveUrlLoader는 **웹 사이트 구조 전체를 크롤링하는 전용 수집기**에 가깝다.
  ```bash
  시작 URL
  ├─ 내부 링크 1
  │   ├─ 내부 링크 1-1
  │   └─ 내부 링크 1-2
  ├─ 내부 링크 2
  └─ 내부 링크 3
  ```
위 구조일때 무든 페이지를 재귀적으로 수집한다.
- 주요 파라미터
  - **url**: 시작 url
  - **max_depth**
    - 링크를 몇 단계 **깊이** 까지 따라갈지 제한
    - 사이트 폭주를 막기 위한 안전장치
      - **0**: 시작페이지만, **1**: 시작페이지 + 1차링크, **2**(기본값): 시작페이지 + 1차링크 + 2차링크
  - **exclude_dirs**: list[str]
    - 크롤링 제외 경로
    - ex) `exclude_dirs=['/login', 'signup']`
  - **prevent_outside**: bool
    - True: base_url 바깥 링크는 가져오지 않고 무시한다.
  - **base_url**: str
    - prevent_outside=True일 때 바깥링크의 기준. 없으면 `url`(시작 url)의 host가 된다. 
  - **extractor**
    - 문서 내용 추출 사용자 정의 함수
    - default는 응답 받은 페이지를 `BeautifulSoup(응답페이지).get_text()` 로 텍스트를 추출한다.
    - ````python
        def custom_extractor(html:str) ->str:
            # 웹 페이지 문서를 입력으로 받는다.
            soup = BeautifulSoup(html, 'lxml')
            return soup.select_one('article').get_text() # 원하는 항목을 추출해서 반환한다.
        
        loader = RecursiveUrlLoader(
            url=start_url,
            extractor=custom_extractor
        )    
    ```

In [ ]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader

def extractor(html:str)->str:
    
    soup = BeautifulSoup(html, "lxml")
    body = soup.select_one("div.body")
    return body.get_text(strip=True, separator="\n") if body else soup.get_text(strip=True, separator="\n")

url = "https://docs.python.org/3"
loader = RecursiveUrlLoader(
    url=url,
    extractor=extractor,
    max_depth=2, 
    prevent_outside=True,
    base_url=url
)
docs = loader.load()

In [ ]:
print(len(docs))

In [ ]:
idx = 10
pprint(docs[idx].metadata)

In [ ]:
print(docs[idx].page_content)

### <del>ArxivLoader</del>

- arxiv api가 업데이트 되고 ArxivLoader는 그에 맞춰 업데이트가 되지 않아 ArxivLoader는 정상적으로 실행되지 않는다. **arxiv api 를 이용해서 검색 후 pdf를 다운로드 받는다.**
- arxiv API: https://github.com/lukasschwab/arxiv.py
- [arXiv-아카이브](https://arxiv.org/) 는 미국 코렐대학에서 운영하는 **무료 논문 저장소**로, 물리학, 수학, 컴퓨터 과학, 생물학, 금융, 경제 등 **과학, 금융 분야의 논문**들을 공유한다.
- 설치
  - `pip install arxiv`

In [ ]:
!uv pip install arxiv

In [ ]:
####################################
#  arxiv lib 사용
####################################
import arxiv

search = arxiv.Search(
    query="Advanced RAG",
    max_results=5,
    sort_by=arxiv.SortCriterion.LastUpdatedDate
)


client = arxiv.Client()
results = client.results(search) # 검색 (iterator)

In [ ]:
print(type(results))

In [ ]:
# 첫번째 것만 조회
paper = next(results)

# 논문 정보
print(paper.title)# 제목
print(paper.authors) # 논문 저자
print(paper.authors[0].name) # Author.name : 이름 추출
print(paper.summary) # 논문 요약(초록)
print(paper.pdf_url) # arivx의 논문 URL
print(paper.get_short_id()) # arivx 의 이 논문의 ID

In [ ]:
# 논문 PDF 저장 
import os 
import requests

save_path = 'data/papers'
os.makedirs(save_path, exist_ok=True)

resp = requests.get(paper.pdf_url)
if resp.status_code == 200:
    with open(os.path.join(save_path, paper.get_short_id()+".pdf"), 'wb') as fo:
        fo.write(resp.content)

In [ ]:
##############################
# 논문 download 함수
##############################

import os 
import requests
def download_arxiv_paper(paper:arxiv.Result, dirpath: str):
    """검색 결과 Paper를 dir_path에 다운로드 후 저장"""
    os.makedirs(dirpath, exist_ok=True)
    resp = requests.get(paper.pdf_url)
    if resp.status_code == 200:
        with open(os.path.join(dirpath, paper.get_short_id()+".pdf"), 'wb') as fo:
            fo.write(resp.content)

In [ ]:
# 전체 다운로드
save_path = 'data/papers'
for paper in results:
    download_arxiv_paper(paper, save_path)

In [ ]:
########################################################################
# arxiv와 pymupdf 를 이용해서 논문을 다운 받은 후 Document list로 반환
########################################################################
import arxiv
import pymupdf                          
from langchain_core.documents import Document
import tempfile, os

def load_arxiv_docs(query: str, top_k: int = 10, dirpath: str=".") -> list[Document]:
    """arxiv에서 논문을 다운 받아서 저장하고 Document로 생성하여 반환.
    Document의 content에는 논문내용을 저장한다.
    Document metadata는 다음 항목을 저장한다.
    - title: 논문제목
    - authors: 논문 저자
    - published: 논문 발표 일자
    - arxiv_url: 논문의 arxiv 사이트 url
    - pdf_url: 논문 PDF 파일 url

    Args:
        query (str): 논문 검색어
        top_k (int, optional): 저장할 논문 편수. Defaults to 10.
        dirpath (str, optional): 논문들을 저장할 디렉토리. Defaults to 현재디렉토리.

    Returns:
        list[Document]: _description_
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=top_k,
        sort_by=arxiv.SortCriterion.Relevance
    )

    docs = []
    for result in client.results(search):
        try:
            # PDF 다운로드 및 텍스트 추출
            download_arxiv_paper(result, dirpath)
            pdf_path = os.path.join(dirpath, result.get_short_id()+".pdf")
            with pymupdf.open(pdf_path) as pdf:
                text = "".join(page.get_text() for page in pdf)

            docs.append(Document(
                page_content=text,
                metadata={
                    "title":     result.title,
                    "authors":   [a.name for a in result.authors],
                    "published": str(result.published),
                    "arxiv_url": result.entry_id,
                    "pdf_url":   result.pdf_url,
                }
            ))
        except Exception as e:
            print(f"[SKIP] {result.title}: {e}")
            continue

    return docs

docs = load_arxiv_docs("Advanced RAG", top_k=10)
print(f"로드된 문서 수: {len(docs)}")


In [ ]:
print(len(docs))

In [ ]:
docs[0].metadata

In [ ]:
print(docs[0].page_content)

### Docling
- IBM Research에서 개발한 오픈소스 문서처리 도구로 다양한 종류의 문서를 구조화된 데이터로 변환해 생성형 AI에서 활용할 수있도록 지원한다.
- **주요기능**
  - PDF, DOCX, PPTX, XLSX, HTML, 이미지 등 여러 형식을 지원
  - PDF의 **페이지 레이아웃, 읽기 순서, 표 구조, 코드, 수식** 등을 분석하여 정확하게 읽어들인다.
  - OCR을 지원하여 스캔된 PDF나 이미지에서 텍스트를 추출할 수있다.
  - 읽어들인 내용을 markdown, html, json등 다양한 형식으로 출력해준다.
- 설치 : `pip install langchain-docling ipywidgets -qU` 
- 참조
  - docling 사이트: https://github.com/docling-project/docling
  - 랭체인-docling https://python.langchain.com/docs/integrations/document_loaders/docling/

In [ ]:
# !uv pip install langchain-docling torch transformers accelerate ipywidgets

## 딥러닝 모델 사용. gpu가 있을 경우 torch cuda 버전을 먼저 설치

In [ ]:
# huggingface 로그인
import os
from dotenv import load_dotenv
from huggingface_hub import login

hf_key = os.getenv("HUGGINGFACE_API_KEY")
login(hf_key)

In [ ]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# PDF 처리 옵션
pdf_options = PdfPipelineOptions()
pdf_options.do_ocr = False  # OCR 끄기 (이유:파이토치와 docling 사용 RapidOCR 호환성 문제 때문)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pdf_options
        )
    }
)

path = "data/papers/2401.15391v1.pdf"
loader = DoclingLoader(
    file_path=path,
    export_type=ExportType.MARKDOWN,
    converter=converter
)
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
print(docs[0].metadata)

In [ ]:
print(docs[0].page_content)

In [ ]:
from IPython.display import Markdown

Markdown(docs[0].page_content)

# Chunking (문서 분할)

![rag_split](figures/rag_split.png)

- Load 한 문서를 지정한 기준의 덩어리(chunk)로 나누는 작업을 진행한다.

## 나누는 이유
1. **임베딩 모델의 컨텍스트 길이 제한**
    - 대부분의 언어 모델은 한 번에 처리할 수 있는 토큰 수에 제한이 있다. 전체 문서를 통째로 입력하면 이 제한을 초과할 수 있어 처리가 불가능해진다.
2. **검색 정확도 향상**
    - 큰 문서 전체보다는 특정 주제나 내용을 다루는 작은 chunk가 사용자 질문과 더 정확하게 매칭된다. 예를 들어, 100페이지 매뉴얼에서 특정 기능에 대한 질문이 있을 때, 해당 기능을 설명하는 몇 개의 문단만 검색되는 것이 더 효과적이다.
    - 사용자 질문에 대해 문서의 모든 내용이 다 관련있는 것은 아니다. Chunking을 통해 가장 관련성 높은 부분만 선별적으로 활용할 수 있어 답변의 품질이 향상된다.
    - 전체 문서에는 질문과 무관한 내용들이 많이 포함되어 있어 모델이 혼란을 겪을 수 있다. 적절한 크기의 chunk는 이런 노이즈를 줄여준다.
3. **계산 효율성**
    - 벡터 유사도 계산, 임베딩 생성 등의 작업이 작은 chunk 단위로 수행될 때 더 빠르고 효율적이다. 메모리 사용량도 줄일 수 있다.

## 주요 Splitter
- **Splitter**는 문서를 분할(chunking)을 처리해주는 도구들이다. Langchain은 분할 대상, 방법에 따라 다양한 splitter를 제공한다.
- **Splitter 의 목표**
  - 가능한 한 **의미 있는 덩어리를 유지**하면서, **최대 길이(chunk_size)**를 넘지 않도록 나누기.
- https://reference.langchain.com/python/langchain_text_splitters/

### CharacterTextSplitter
가장  기본적인 Text spliter
- 한개의 구분자를 기준으로 분리한다. (default: "\n\n")
    - 분리된 조각이 chunk size 보다 작으면 다음 조각과 합칠 수 있다.
        - 합쳤을때 chuck_size 보다 크면 안 합친다. chuck_size 이내면 합친다.
    - 나누는 기준은 구분자이기 때문에 chunk_size 보다 글자수가 많을 수 있다.
- chunk size: 분리된 문서(chunk) 글자수 이내에서 분리되도록 한다.
    -  구분자를 기준으로 분리한다. 구분자를 기준으로 분리한 문서 조각이 chunk size 보다 크더라도 그대로 유지한다. 즉 chunk_size가 우선이 아니라 **seperator** 가 우선이다.
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - seperator: 구분 문자열을 지정. (default: '\n\n')
- CharacterTextSplitter는 단순 스플리터로 overlap기능을 지원하지는 않는다. 단 seperator가 빈문자열("") 일 경우에는 overlap 기능을 지원한다. overlap이란 각 이전 청크의 뒷부분의 문자열을 앞에 붙여 문맥을 유지하는 것을 말한다.
  
### RecursiveCharacterTextSplitter
- RecursiveCharacterTextSplitter는 **긴 텍스트를 지정된 최대 길이(chunk_size) 이하로 나누는 데 효과적인 텍스트 분할기**(splitter)이다.
- 여러 **구분자(separators)를 순차적으로 적용**하여, 가능한 한 자연스러운 문단/문장/단어 단위로 분할하고, 최종적으로는 크기 제한을 만족시킨다.
- 분할 기준 문자
    1. 두 개의 줄바꿈 문자 ("\n\n")
    2. 한 개의 줄바꿈 문자 ("\n")
    3. 공백 문자 (" ")
    4. 빈 문자열 ("")
- 작동 방식
    1. 먼저 가장 높은 우선순위의 구분자("\n\n")를 기준으로 분리한다.
    2. 분할된 조각 중 **chunk_size를 초과하는 조각**에 대해 다음 우선순위 구분자("\n" → " " → "")로 재귀적으로 재분할한다.
    3. 이 과정을 통해 모든 조각(chunk)이 chunk_size를 초과하지 않도록 만든다.  
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - chunk_overlap: 연속된 청크들 간의 겹치는 문자 수를 설정. 새로운 청크 생성 시 이전 청크의 마지막 부분에서 지정된 수만큼의 문자를 가져와서 새 청크의 앞부분에 포함시켜, 청크 경계에서 문맥의 연속성을 유지한다.
      - 구분자에 의해 청크가 나눠지면 정상적인 분리이므로 overlap이 적용되지 않는다.
      - 정상적 구분자로 나눌 수 없어 chunk_size에 맞춰 잘라진 경우 문맥의 연결성을 위애 overlap을 적용한다.
    - separators(list): 구분자를 지정한다. 지정하면 기본 구분자가 지정한 것으로 변경된다.

#### 메소드
- `split_documents(Iterable[Document]) : List[Document]`
    - Document 목록을 받아 split 처리한다.
- `split_text(str) : List[str]`
    - string text를 받아서 split 처리한다. 

In [ ]:
# !uv pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

text = """123456789012345678901234567890123456789012345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""
splitter = CharacterTextSplitter(
    chunk_size=60,
    chunk_overlap=10,
    # separator=""
)
result = splitter.split_text(text)
print(len(result))

In [ ]:
for txt in result:
    print(len(txt), txt, sep="-")
    print("-----------------------------------")

In [ ]:
doc = Document(page_content=text, metadata={"category":"split"})

docs2 = splitter.split_documents([doc])
len(docs2)

In [ ]:
for doc in docs2:
    print(len(doc.page_content), doc.page_content)

In [ ]:
len("""abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ RSTUVWXYZ""")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text2 = """1234567890123456789012345678901234567890
12345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ RSTUVWXYZ
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
    seperators=["\n\n", '\n', '[.?!,]', ' ', ''],
    is_separator_regex=True
)
result2 = splitter.split_text(text2)

for txt in result2:
    print(len(txt), txt, sep="-")
    print("-------------------------------")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
path = "data/olympic.txt"

# Loading -> Split
loader = TextLoader(path, encoding="utf-8")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=20
)

load_docs = loader.load()
docs = splitter.split_documents(load_docs)
docs = [doc for doc in docs if len(doc.page_content) > 10]
len(docs)

In [ ]:
print(docs[0].page_content)

In [ ]:
docs = loader.load_and_split(splitter) # splitter를 이용해서 load 와 split을 한번에 처리.
len(docs)

## Token 수 기준으로 나누기

- LLM 언어 모델들은 입력 토큰 수 제한이 있어서 요청시 제한 토큰수 이상의 프롬프트는 전송할 수 없다.
- 따라서 텍스트를 chunk로 분할할 때는 글자수 보다 **토큰 수를 기준으로 크기를 지정하는 것**이 좋다.  
- 토큰기반 분할은 텍스트의 의미를 유지하면서 분할하는 방식이므로 문자 기반 분할과 같이 단어가 중간잘리는 것들을 방지할 수 있다. 
- 토큰 수 계산할 때는 사용하는 언어 모델에 사용된 것과 동일한 tokenizer를 사용하는 것이 좋다.
  - 예를 들어 OpenAI의 GPT 모델을 사용할 경우 tiktoken 라이브러리를 활용하여 토큰 수를 정확하게 계산할 수 있다.

### [tiktoken](https://github.com/openai/tiktoken) tokenizer 기반 분할
- OpenAI에서 GPT 모델을 학습할 때 사용한 `BPE` 방식의 tokenizer. **OpenAI 언어모델을 사용할 경우 이것을 사용하는 것이 좀 더 정확하게  토큰을 계산할 수 있다.**
- Splitter.from_tiktoken_encoder() 메소드를 이용해 생성
  - `RecursiveCharacterTextSplitter.from_tiktoken_encoder()`
  - `CharacterTextSplitter.from_tiktoken_encoder()`
- 파라미터
  - encode_name: 인코딩 방식(토큰화 규칙)을 지정. OpenAI는 GPT 모델들 마다 다른 방식을 사용했다. 그래서 사용하려는 모델에 맞는 인코딩 방식을 지정해야 한다.
    - `o200k_base`: GPT-4 이후 모델들이 사용한 방식
    - `cl100k_base`: 초기 GPT-4 및 GPT-3.5-Turbo 모델에서 사용된 방식.
    - `r50k_base:` GPT-3 모델에서 사용된 방식 
  - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- tiktoken 설치
  - `pip install tiktoken`

### HuggingFace Tokenizer
- HuggingFace 모델을 사용할 경우 그 모델이 사용한 tokenizer를 이용해 토큰 기반으로 분할 한다.
  - 다른 tokenizer를 이용해 분할 할 경우 토큰 수 계산이 다르게 될 수있다.
- `from_huggingface_tokenizer()` 메소드를 이용.
  - 파라미터
    - tokenizer: HuggingFace tokenizer 객체
    - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- `transformers` 라이브러리를 설치해야 한다.
  - `pip install transformers` 

In [ ]:
# !uv pip show tiktoken
# !uv pip show transformers

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"
loader = TextLoader(path, encoding="utf-8")
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
     model_name="gpt-5-mini",
     chunk_size=500,
     chunk_overlap=20
)
docs = loader.load_and_split(splitter)
len(docs)

In [ ]:
print(docs[0].page_content)

In [ ]:
from transformers import AutoTokenizer
model_id = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(type(tokenizer))

In [ ]:
splitter2 = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=500,
    chunk_overlap=20
)

docs = loader.load_and_split(splitter2)
len(docs)

## MarkdownHeaderTextSplitter
- Markdown Header 기준으로 Splitter
- Loading한 문서가 Markdown 문서이고 Header를 기준으로 문서의 내용이 나눠질때 사용.
- https://reference.langchain.com/python/langchain_text_splitters/#langchain_text_splitters.MarkdownTextSplitter

In [ ]:
text = """
# 대주제1
- 동물

## 중주제1
- 포유류

- 조류

### 소주제1
- 개
- 고양이
- 까치
- 독수리

# 대주제2
## 중주제2
- 기차
- 배
"""

In [ ]:
# <h1> ~ <h6> # ~ #*6
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split = [
    ("#", "header1"),
    ("##", "header2"),
    ("###", "header3")
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split,
    strip_headers=True,
    return_each_line=False
)

docs = splitter.split_text(text)
len(docs)

In [ ]:
for doc in docs:
    print(doc.metadata)
    print(doc.page_content)
    print("======================================")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

path = "data/olympic_wiki.md"

loader = TextLoader(path, encoding="utf-8")
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3")
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
)

In [ ]:
# docs = loader.load()
# doc_txt = '\n'.join(doc.page_content for doc in docs)

In [ ]:
split_docs = splitter.split_text(doc_txt)
len(split_docs)

In [ ]:
split_docs[0]